In [1]:
import pandas as pd
import yaml
from pathlib import Path
import joblib
import useful_funcs

In [2]:
# ========== Opening config file ==========

path_yaml = Path("config.yaml")
try:

    with open(path_yaml, "r") as file:
        config = yaml.safe_load(file)

except FileNotFoundError:
    print("Config file not found")
    sys.exit(1)

In [12]:
# ========== creating dataframe, removing clients with fiber, creating path variables ==========

path_processed = Path(config["paths"]["processed"])

path_model_fiber = Path(config["paths"]["models"]).joinpath("fiber.pkl")
path_model_str_tv = Path(config["paths"]["models"]).joinpath("str_tv.pkl")
path_model_str_movies = Path(config["paths"]["models"]).joinpath("str_movies.pkl")

path_predictions = Path(config["paths"]["predictions"])


df = pd.read_csv(path_processed, index_col="customer_id")

In [14]:
# ========== Creating predicting functions ==========

def predict_fiber(df, model_path, save_csv=False):
    df_nofiber = df[df["is_fiber"] == 0].drop(columns="is_fiber")
    
    # ========== loading model and creating predictions ==========
    
    model_fiber = joblib.load(model_path)
    
    pred_prob = model_fiber.predict_proba(df_nofiber)[:, 1]

    # ========== creating prediction dataframe and exporting as a csv ==========
    
    df_prediction_fiber = pd.DataFrame({
        "Fiber": pred_prob
    }, index=df_nofiber.index).sort_values(by="Fiber", ascending=False)

    if save_csv:
        df_prediction_fiber.to_csv(path_predictions.joinpath("fiber.csv"), index="customer_id")
    
    return df_prediction_fiber

In [15]:
def predict_str_tv(df, model_path, save_csv=False):
    df_no_str_tv = df[df["streaming_tv"] == 0].drop(columns="streaming_tv")

    # ========== loading model and creating predictions ==========
    
    model_str_tv = joblib.load(model_path)
    
    pred_prob = model_str_tv.predict_proba(df_no_str_tv)[:, 1]

    # ========== creating prediction dataframe and exporting as a csv ==========
    
    df_prediction_str_tv = pd.DataFrame({
        "Streaming TV": pred_prob
    }, index=df_no_str_tv.index).sort_values(by="Streaming TV", ascending=False)
    
    if save_csv:
        df_prediction_str_tv.to_csv(path_predictions.joinpath("str_tv.csv"), index="customer_id")
    
    return df_prediction_str_tv

In [19]:
def predict_str_movies(df, model_path, save_csv=False):
    df_no_str_movies = df[df["streaming_movies"] == 0].drop(columns="streaming_movies")

    # ========== loading model and creating predictions ==========
    
    model_str_movies = joblib.load(model_path)
    
    pred_prob = model_str_movies.predict_proba(df_no_str_movies)[:, 1]

    # ========== creating prediction dataframe and exporting as a csv ==========
    
    df_prediction_str_movies = pd.DataFrame({
        "Streaming Movies": pred_prob
    }, index=df_no_str_movies.index).sort_values(by="Streaming Movies", ascending=False)
    
    if save_csv:
        df_prediction_str_movies.to_csv(path_predictions.joinpath("str_movies.csv"), index="customer_id")
    
    return df_prediction_str_movies

In [20]:
predict_fiber(df, path_model_fiber, True)

,Fiber
customer_id,
2985-JUUBZ,0.955397
2294-DMMUS,0.939218
1697-BCSHV,0.917932
6217-TOWGS,0.914572
0292-WEGCH,0.914243
...,...
0621-JFHOL,0.004275
0486-LGCCH,0.004102
1494-EJZDW,0.003956


In [21]:
predict_str_tv(df, path_model_str_tv, True)

,Streaming TV
customer_id,
4686-GEFRM,0.890084
8468-FZTOE,0.889630
7159-NOKYQ,0.887943
0853-TWRVK,0.877104
7730-CLDSV,0.870726
...,...
8410-BGQXN,0.012804
8048-DSDFQ,0.011949
9753-OYLBX,0.011949


In [22]:
predict_str_movies(df, path_model_str_movies, True)

,Streaming Movies
customer_id,
1937-OTUKY,0.879437
7663-YJHSN,0.878173
1452-UZOSF,0.875105
7005-CYUIL,0.869027
5668-MEISB,0.867305
...,...
2005-DWQZJ,0.033616
2520-SGTTA,0.033616
3525-DVKFN,0.033616
